# N02: VLM descriptions

The baseline fails on what the metadata does not say. Here a vision-language model (`qwen/qwen3-vl-8b-instruct` via
OpenRouter, temperature 0) writes 2 sentences about each painting, once, offline. At query time only the visitor's
question is encoded, so the kiosk does not need the VLM.

- R2 searches the description only (one change from R1: metadata -> visual text).
- R3 searches description + metadata.

Cost: about 1,080 images (dev + test; train is not used) at about 64 image tokens each, roughly $0.10-0.30.

In [ ]:
import pandas as pd

from deep_learning_project_waerniers_lamy.nlp import utils as u

pd.set_option("display.width", 160)
import os
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

## 1. Generate the descriptions (once)

Needs an OpenRouter key in the environment (`$env:OPENROUTER_API_KEY = "sk-or-..."` in PowerShell before starting
Jupyter). Resumable: already described paintings are skipped. Try `limit=5` first and read them.

In [ ]:
assert os.environ.get("OPENROUTER_API_KEY"), "set OPENROUTER_API_KEY first"
descriptions = u.generate_descriptions(limit=5)

In [ ]:
descriptions = u.generate_descriptions()  # all remaining dev + test paintings

## 2. Look at a few

In [ ]:
paintings = u.load_corpus("dev")
descriptions = u.load_descriptions()
sample = paintings.sample(4, random_state=u.SEED)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (_, row) in zip(axes, sample.iterrows()):
    ax.imshow(Image.open(u.DATA_DIR / row["path"]))
    ax.set_title(row["genre"], fontsize=9)
    ax.axis("off")
plt.show()
for _, row in sample.iterrows():
    print(f"{row['id']}: {descriptions[row['id']]}")

Two checks on the descriptions:
- length: the prompt asks for 2 sentences (ADR 0005), cut at 60 tokens;
- **artist leak**: if the VLM recognises famous works and names the artist, R2 would partly be using metadata, not
  only what it sees.

In [ ]:
words = pd.Series([len(descriptions[i].split()) for i in paintings["id"]])
print("words per description:", words.describe().round(1).to_dict())
print("artist named in the description:", round(u.artist_leak_rate(paintings, descriptions), 3))

## 3. Rank, judge the new top-5 paintings

In [ ]:
queries = u.QUERIES["text"].tolist()
scores = {
    "R0 random": u.rung_random(paintings, queries),
    "R1 metadata": u.rung_metadata(paintings, queries),
    "R2 description": u.rung_description(paintings, queries, descriptions),
    "R3 description+metadata": u.rung_description_metadata(paintings, queries, descriptions),
}
u.write_judging_page("dev", scores, paintings)

### Judging (by hand)

Open the page printed above in a browser. It shows **only the query and the image** (no rung name, artist, style or
description), and only paintings not judged before. Grade each one 0 / 1 / 2 with the keyboard, click *export JSON*,
then run the next cell. Grades are kept in the browser, so you can stop and come back.

In [ ]:
u.import_judgments("dev", Path.home() / "Downloads" / "judgments_dev.json")

## 4. Results

Earlier rungs are re-scored with all judgments so far: the ideal ranking of a query can only improve as more relevant
paintings are found, so their nDCG may move slightly. The question of this step: **does visual text beat metadata?**
(R2 vs R1, paired over the same queries; pre-registered comparison 1 of 3.)

In [ ]:
per_query = u.per_query_scores("dev", scores, paintings)
table = u.summary_table(per_query)
u.plot_ladder(table, "dev: + VLM descriptions")
display(table)
u.compare(per_query, [("R2 description", "R1 metadata")])

In [ ]:
per_query.groupby(["rung", "category"])["ndcg@5"].mean().unstack().round(3)